# V8 lightweight repeat: all readers, unseen sentences
Same scientific setup as the completed-but-unsaved V8:4 fits, seeds45/62, real/arbitrary labels; all12 readers remain separate examples.66/22/22 disjoint sentence IDs;792 fit /264 validation /264 test. NO reader averaging. Original CNN/raw TRT/LR0.0001/settings unchanged.
Save predictions, scores, hyperparameters and small audit/history records. NO saved weights, datasets or large models. Expected output well below a few MB. New output folder preserves old run. One split and one arbitrary mapping; exploratory, no significance claim.


## 1. GPU setup
Choose a GPU runtime. Fixed source tag: `v7-v8-light-results-v1`. No package installation or data download is performed; existing Drive inputs are reused.

In [ ]:
import os, sys, subprocess, json
import tensorflow as tf
assert tf.config.list_physical_devices('GPU'), 'Select a GPU runtime and rerun setup.'
from google.colab import drive
drive.mount('/content/drive')
repo = '/content/teco-v8-light-results-code'
run_ref = 'v7-v8-light-results-v1'
if not os.path.isdir(repo):
    subprocess.run(['git', 'clone', '-q', '--branch', run_ref, '--single-branch',
                    'https://github.com/parmisbathayan/reproduce-teco-results.git', repo], check=True)
os.chdir(repo)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
assert commit == subprocess.check_output(['git', 'rev-parse', run_ref + '^{commit}'], text=True).strip()
assert not subprocess.check_output(['git','status','--porcelain'], text=True).strip(), 'Use a clean fixed revision.'
base = '/content/drive/MyDrive/Thesis/Results/reproduce_yaser_codes/eegonly'
mirror_dir = os.path.join(base, 'v8_light_seen_readers_45_62')
out_dir = '/content/teco-v8-light-results'
command = [sys.executable, '-u', '-B', 'src/v8_seen_readers_light.py',
           '--trt-dir', '/content/drive/MyDrive/Thesis/Data/TRT_Total',
           '--labels-csv', '/content/drive/MyDrive/Thesis/Data/teco_sentiment_labels_task1.csv',
           '--mirror-dir', mirror_dir, '--out-dir', out_dir, '--config', 'configs/v8_seen_readers_unseen_sentences.json',
           '--reference-out-dir', os.path.join(base, 'v4_low_lr_confirmation_45_62'),
           '--v7-out-dir', os.path.join(base, 'v7_unseen_sentences_45_62')]
def stream_run(arguments):
    with subprocess.Popen(arguments, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
            returncode = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            print('Stopped. Verified job ZIPs remain saved; rerun setup, preflight and training to resume.')
            raise
    if returncode:
        raise subprocess.CalledProcessError(returncode, arguments)
print('GPU ready; fixed Git commit:', commit, flush=True)

from pathlib import Path
from uuid import uuid4
Path(mirror_dir).mkdir(parents=True, exist_ok=True)
probe = Path(mirror_dir)/(str(uuid4()) + '_storage_probe.txt')
probe.write_text('Light V8 storage check: confirm this text on Drive website.')
print('CHECK ON DRIVE WEBSITE:', probe.name, probe.stat().st_size, 'bytes')
print('If the shared Thesis folder is absent from My Drive, add a shortcut or edit the paths.')


## 2. Preflight — no training
Checks model, data and runtime against V4 and V7; requires the identical saved sentence split and label mapping; verifies sentence text/ID alignment across all readers and rejects exact nonzero-array overlap across sentence partitions. Existing V7 files are read only. Stop on any error; do not bypass checks.

In [ ]:
stream_run(command + ['--prepare-only'])
print('Expected: 792 training examples, 264 validation, 264 test; exactly balanced labels. No readers averaged.')


## 3. Confirm storage, then run or resume4 fits
In the account mounted in Colab, find the new storage probe on the Drive WEBSITE and confirm its text/nonzero size. Then set the flag True below. Folder access alone is insufficient. About10–20 GPU minutes, estimate. Results save after each fit; no model weights retained.


In [ ]:
DRIVE_PERSISTENCE_CHECKED = False
assert DRIVE_PERSISTENCE_CHECKED, 'Confirm the probe on the Drive website in the mounted account first.'
stream_run(command + ['--storage-checked'])


## 4. Verify and show results — no training
Recompute metrics from saved predictions. Confirm four ZIPs and summaries appear with nonzero sizes on the Drive website before disconnecting. Local primary files remain under /content until runtime deletion.


In [ ]:
stream_run(command + ['--verify-only'])
import pandas as pd
from IPython.display import display
with open(os.path.join(out_dir, 'completion.json')) as stream:
    completion = json.load(stream)
assert completion['complete'] and completion['completed_fits'] == 4
display(pd.read_csv(os.path.join(out_dir, 'summary_by_seed.csv')))
display(pd.read_csv(os.path.join(out_dir, 'summary_overall.csv')))
print('Four fitted models verified, all 1,056 test predictions retained in jobs/*.zip. Only 22 unique test sentence identities.')
print('No additional experiment is scheduled automatically. Send these results for review.')
from pathlib import Path
print('Actual local results bytes:', sum(p.stat().st_size for p in Path(out_dir).rglob('*') if p.is_file()))
print('No model weights saved. Confirm all four job ZIPs and summaries on the Drive WEBSITE before disconnecting.')
